In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:

# ============================================
# FREECODECAMP BOOK RECOMMENDATION ENGINE
# K-Nearest Neighbors (KNN)
# ============================================

import pandas as pd
from sklearn.neighbors import NearestNeighbors

# 1. Keep users who have rated at least 200 books
user_counts = df_ratings['user'].value_counts()
active_users = user_counts[user_counts >= 200].index

ratings = df_ratings[
    df_ratings['user'].isin(active_users)
].copy()

# 2. Keep books that have received at least 100 ratings
book_counts = ratings['isbn'].value_counts()
popular_books = book_counts[book_counts >= 100].index

ratings = ratings[
    ratings['isbn'].isin(popular_books)
].copy()

# 3. Merge ratings with book information
merged_data = pd.merge(
    ratings,
    df_books,
    on='isbn'
)

# 4. Create the user-book rating matrix
book_user_matrix = merged_data.pivot_table(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

# 5. Train the KNN model
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(book_user_matrix.values)

print("Model trained successfully!")
print("Number of books:", book_user_matrix.shape[0])
print("Number of users:", book_user_matrix.shape[1])
print("Model features:", model.n_features_in_)

# 6. Create the recommendation function
def get_recommends(book):

    if book not in book_user_matrix.index:
        raise ValueError(
            f"Book '{book}' is not available in the recommendation dataset."
        )

    distances, indices = model.kneighbors(
        book_user_matrix.loc[book].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []

    for i in range(1, 6):
        recommended_books.append([
            book_user_matrix.index[indices[0][i]],
            distances[0][i]
        ])

    # Return recommendations from greatest to smallest distance
    recommended_books.reverse()

    return [book, recommended_books]


# 7. Test the recommendation engine
print("\nRecommendation test:")

print(
    get_recommends(
        "Where the Heart Is (Oprah's Book Club (Paperback))"
    )
)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):


  return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()